In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm  # for progress bar
from torch.cuda.amp import GradScaler, autocast
from datetime import datetime

from TtoGmodel_10 import TextToGraphTransformer

In [2]:
from Circuits import Circuits
circuits= Circuits()

Loading dataset files...
Loaded dataset files successfully.


In [3]:
print(circuits.component_lists[0])
print(circuits.graphs[0])

['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
[[0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 1 1 1 1]
 [1 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0]
 [0 0 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0]]


In [4]:
def collate_fn(batch):
    seqs, mats = zip(*batch)

    # Convert sequences to torch tensors
    seqs = [torch.tensor(seq, dtype=torch.long) for seq in seqs]

    # Convert adjacency matrices (NumPy -> PyTorch)
    mats = [torch.tensor(mat, dtype=torch.float32) for mat in mats]

    # Get max sizes
    max_seq_len = max(len(seq) for seq in seqs)
    max_nodes = max(mat.size(0) for mat in mats)

    # Pad sequences
    padded_seqs = torch.stack([
        F.pad(seq, (0, max_seq_len - len(seq)), value=0)
        for seq in seqs
    ])

    # Pad adjacency matrices
    padded_mats = torch.stack([
        F.pad(mat, (0, max_nodes - mat.size(1), 0, max_nodes - mat.size(0)), value=0)
        for mat in mats
    ])

    seq_lengths = torch.tensor([len(seq) for seq in seqs])

    return padded_seqs, padded_mats, seq_lengths


In [5]:
from sklearn.model_selection import train_test_split

dataset = list(zip(circuits.component_indices, circuits.graphs))
# Split dataset into training and testing sets
train_data, test_data = train_test_split(dataset, test_size=0.1, random_state=42)
loader = torch.utils.data.DataLoader(train_data, batch_size=12, shuffle=True, collate_fn=collate_fn)


In [6]:
print(circuits.component_indices[0])
print(circuits.graphs[0])


[318, 376, 356, 425, 469, 189, 672, 341]
[[0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 1 1 1 1]
 [1 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0]
 [0 0 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0]]


In [7]:
# Initialize model parameters
vocab_size = len(circuits.vocab)  # Number of unique components
embedding_dim = 128
hidden_dim = 256
num_heads = 8
num_layers = 4
dropout = 0.1

# Initialize the model
model = TextToGraphTransformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

TextToGraphTransformer(
  (embedding): Embedding(892, 128, padding_idx=0)
  (positional_encoding): SinusoidalPositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): R

In [ ]:
def calculate_test_accuracy(model, test_data, device, threshold=0.5):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for input_seq, true_adj in test_data:
            input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
            seq_len = input_tensor.size(1)
            seq_mask = (input_tensor != PAD_TOKEN_ID)

            logits = model(input_tensor, seq_mask)
            probs = torch.sigmoid(logits.squeeze(0))  # Shape: (S, S)
            predicted_adj = (probs > threshold).float().cpu().numpy()

            # Compare predicted adjacency matrix with the ground truth
            correct += (predicted_adj == true_adj).sum()
            total += true_adj.size

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")


In [10]:
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler

PAD_TOKEN_ID = 0
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()  # For mixed-precision training (optional)

num_epochs = 100
dataloader = loader
save_every = 10 # Save every N epochs

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}", leave=False)

    for input_seqs, adj_mats, seq_lengths in progress_bar:
        input_seqs = input_seqs.to(device, non_blocking=True)
        adj_mats = adj_mats.to(device, non_blocking=True)
        seq_lengths = seq_lengths.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():  # Mixed-precision forward pass (optional)
            seq_mask = (input_seqs != PAD_TOKEN_ID)  # [B, S]
            predicted_logits = model(input_seqs, seq_mask)


            # Efficient masking (flatten first)
            mask = (input_seqs != PAD_TOKEN_ID)
            mask2d = mask.unsqueeze(2) & mask.unsqueeze(1)
            mask_flat = mask2d.view(-1)                   # [B*S*S]
            pred_flat = predicted_logits.view(-1)
            true_flat = adj_mats.view(-1)

            loss = criterion(pred_flat[mask_flat], true_flat[mask_flat])

        # Backprop with mixed precision
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach().item()  # Detach to avoid memory buildup
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader)
    print(f"[Epoch {epoch}] Avg Loss: {avg_loss:.4f}")
    calculate_test_accuracy(model, test_data, device)
    if epoch % save_every == 0:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"./Saves/toGmodel_epoch{epoch}_{timestamp}.pth"
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab_size': vocab_size,
            'embedding_dim': embedding_dim,
            'hidden_dim': hidden_dim,
            'num_heads': num_heads,
            'num_layers': num_layers,
            'dropout': dropout,
            'learning_rate': 1e-4,
        }
        torch.save(checkpoint, save_path)
        print(f"💾 Checkpoint saved at epoch {epoch} → {save_path}")

# Save model
torch.save(model.state_dict(), 'TextToGraphTransformer.pth')


C:\Users\MSI\AppData\Local\Temp\ipykernel_1992\2568644609.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # For mixed-precision training (optional)
Epoch 1:   0%|          | 0/252 [00:00<?, ?it/s]C:\Users\MSI\AppData\Local\Temp\ipykernel_1992\2568644609.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # Mixed-precision forward pass (optional)
c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


[Epoch 1] Avg Loss: 0.1396
Test Accuracy: 0.9737


[Epoch 2] Avg Loss: 0.1161
Test Accuracy: 0.9737


[Epoch 3] Avg Loss: 0.1061
Test Accuracy: 0.9742


[Epoch 4] Avg Loss: 0.0766
Test Accuracy: 0.9854


...

[Epoch 27] Avg Loss: 0.0224
Test Accuracy: 0.9928


[Epoch 28] Avg Loss: 0.0221
Test Accuracy: 0.9928


[Epoch 29] Avg Loss: 0.0217
Test Accuracy: 0.9929


[Epoch 30] Avg Loss: 0.0215
Test Accuracy: 0.9929
💾 Checkpoint saved at epoch 30 → ./Saves/toGmodel_epoch30_20250422_231214.pth


KeyboardInterrupt: 

In [14]:
from sklearn.metrics import precision_score, confusion_matrix

# Select a sample from test_data
sample_seq, sample_adj = test_data[4]  # Replace 0 with the desired index

# Convert the sample sequence to a tensor
input_tensor = torch.tensor(sample_seq, dtype=torch.long).unsqueeze(0).to(device)  # [1, S]

# Generate the sequence mask
seq_mask = (input_tensor != PAD_TOKEN_ID)

# Evaluate the model
model.eval()
with torch.no_grad():
    logits = model(input_tensor, seq_mask)
    probs = torch.sigmoid(logits.squeeze(0))  # Shape: (S, S)
    predicted_adj = (probs > 0.5).float().cpu().numpy()

# Print the results
print("Input Sequence:", sample_seq)
print("True Adjacency Matrix:")
print(sample_adj)
print("Predicted Adjacency Matrix:")
print(predicted_adj)

# Flatten the true and predicted adjacency matrices for comparison
true_flat = sample_adj.flatten()
predicted_flat = predicted_adj.flatten()

# Calculate precision
precision = precision_score(true_flat, predicted_flat, zero_division=0)
print("Precision:", precision)

# Calculate confusion matrix
conf_matrix = confusion_matrix(true_flat, predicted_flat)
print("Confusion Matrix:")
print(conf_matrix)

Input Sequence: [318, 376, 356, 173, 211, 157, 619, 747, 855, 757, 147, 601, 604, 443, 568, 11, 235, 650, 226, 168, 758, 743, 69, 779, 338, 622, 482, 425, 469, 189, 672, 341, 206, 401, 884, 559, 288, 581, 567, 152, 68, 311, 370, 386, 886, 81, 459, 299, 839, 623, 195, 873, 770, 494, 151, 858, 67, 176, 644, 639, 489, 452, 48, 404, 6, 227, 837, 583, 418, 184, 723, 797, 687, 500, 818, 331, 105, 605, 367, 634, 77, 822, 2, 763, 434, 846, 734, 786, 312, 138, 38, 620, 594, 505, 749, 843]
True Adjacency Matrix:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 1 0 0]
 [0 1 0 ... 1 0 0]]
Predicted Adjacency Matrix:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 1. 1.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 1. 0. 0.]]
Precision: 0.9782608695652174
Confusion Matrix:
[[8902    6]
 [  38  270]]


In [ ]:
def evaluate_adjacency_matrix(model, input_seq, vocab, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # [1, S]
        seq_len = input_tensor.size(1)
        dummy_adj = torch.zeros((1, seq_len, seq_len), dtype=torch.float).to(device)
        dummy_lengths = torch.tensor([seq_len]).to(device)  # Sequence length for this input

        # Pass both input_tensor and dummy_lengths to the model
        seq_mask = (input_tensor != PAD_TOKEN_ID)
        logits = model(input_tensor, seq_mask)

        
        probs = torch.sigmoid(logits.squeeze(0))  # Shape: (S, S)
        binary_adj = (probs > threshold).float()

        print("\nPredicted Adjacency Matrix (Binary, N x N):")
        print(binary_adj.cpu().numpy())


In [ ]:
ex_index = 2
sample_input = circuits.component_indices[ex_index]
print("Sample Input Sequence:", sample_input)
print("Sample Input Sequence:", circuits.component_lists[ex_index])
print("Actual Adjacency Matrix:")
print(circuits.graphs[ex_index])
evaluate_adjacency_matrix(model, sample_input, circuits.vocab, device)

Sample Input Sequence: [750, 570, 340, 23, 595, 736, 6, 489, 284, 764, 815, 569, 487, 172, 157]
Sample Input Sequence: ['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Actual Adjacency Matrix:
[[0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 1 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1]
 [0 0 0 1 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 0 0]]

Predicted Adjacency Matrix (Binary, N x N):
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1.]
 [0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1.]
 [0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1.

In [ ]:
def load_checkpoint(path, device='cuda'):
    checkpoint = torch.load(path, map_location=device)

    model = TextToGraphTransformer(
        vocab_size=checkpoint['vocab_size'],
        embedding_dim=checkpoint['embedding_dim'],
        hidden_dim=checkpoint['hidden_dim'],
        num_heads=checkpoint['num_heads'],
        num_layers=checkpoint['num_layers'],
        dropout=checkpoint['dropout']
    ).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer = torch.optim.Adam(model.parameters(), lr=checkpoint['learning_rate'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    print(f"✅ Loaded model from {path} (epoch {checkpoint['epoch']})")
    return model, optimizer, checkpoint['epoch']


In [ ]:
# Example usage

# 1. Path to saved checkpoint
file_name = "TtoGmodel_epoch90_20250418_004110.pth"
checkpoint_path = './Saves/' + file_name # Replace with your file

# 2. Load the model and optimizer
model, optimizer, start_epoch = load_checkpoint(checkpoint_path, device=device)


ex_index = 3300
sample_input = circuits.component_indices[ex_index]
print("Sample Input Sequence:", circuits.component_lists[ex_index])
print("Actual Adjacency Matrix:")
print(circuits.graphs[ex_index])
evaluate_adjacency_matrix(model, sample_input, circuits.vocab, device)



C:\Users\MSI\AppData\Local\Temp\ipykernel_18512\3424363509.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


RuntimeError: Error(s) in loading state_dict for TextToGraphTransformer:
	size mismatch for edge_mlp.0.weight: copying a param with shape torch.Size([256, 257]) from checkpoint, the shape in current model is torch.Size([256, 256]).